# Plan: see what a launch will do before it runs

A plan answers the question "what would Xeda do for this request?". `ws.plan` starts no tool, runs no
generator and writes nothing in a run directory. It is the Python form of `xeda run --dry-run`. This notebook plans a chain,
reads one node of the plan, runs the plan, and shows how a plan goes stale and how a wrong request
fails at planning.

The notebook needs Yosys and `nextpnr-ecp5` on your `PATH`. It writes only under `notebook-output/`
in its working directory.

## Set up

In [ ]:
import os
from pathlib import Path

import xeda
from xeda import Design, Spec, Workspace
from xeda.errors import SettingsError, SpecError, StalePlanError

# The examples of the Xeda checkout. Set XEDA_EXAMPLES to use another copy.
EXAMPLES = Path(os.environ.get("XEDA_EXAMPLES", Path.cwd().parent / "examples")).resolve()
if not EXAMPLES.is_dir():
    raise FileNotFoundError(
        f"No examples directory at {EXAMPLES}. "
        "Set the XEDA_EXAMPLES environment variable to the `examples` directory of an Xeda checkout."
    )

# This notebook writes only under WORK.
WORK = Path("notebook-output").resolve()
ws = Workspace(WORK / "xeda_run", local_settings=xeda.local_settings())
print("Xeda", xeda.__version__)
print("run root:", ws.run_root)

## Plan a spec

`yosys_fpga+nextpnr` is a chain. The first flow writes a netlist and the second flow reads it. The
last flow is the root of the chain: the flow that you asked for.

In [ ]:
design = Design.load(EXAMPLES / "sv/fifo/fifo.xeda.yaml")

spec = Spec(
    "yosys_fpga+nextpnr",
    design,
    settings={
        "yosys_fpga": {"fpga.part": "LFE5U-85F-6BG381C", "clock.period": "10 ns"},
        "nextpnr": {"lpf_allow_unconstrained": True, "seed": 1},
    },
)
plan = ws.plan(spec)
print("roots:", plan.roots)

`plan.nodes` maps each node name to a `PlanNode`, in execution order: a producer comes before its
consumers. The `state` of a node is what a launch would do with it:

- `fresh`: Xeda would reuse the last run. `reason` is empty.
- `rejected`: Xeda would reuse the last run and report its rejection again.
- `stale`: Xeda would run the node. `reason` says why.
- `unknown`: Xeda cannot tell before something is made, for example a producer that would run,
  a generator that has not run, or a prepared input that is not made yet. `reason` says what.
- `blocked`: a producer is rejected or blocked, so the node would not run.

When you run this notebook for the first time, there is no run yet and both nodes are stale.

In [ ]:
for name, node in plan.nodes.items():
    print(f"{name:11} {node.state:6} {node.reason}".rstrip())

A state is an observation at the time of planning, not a promise. A launch checks it again.

## Inspect one node

`plan.nodes[name]` is one `PlanNode`. It has the flow, the run directory, the effective settings
and the types of the files that the node reads and writes. The settings that the flows share, such
as the clocks, came along the chain from the first flow.

In [ ]:
node = plan.nodes["nextpnr"]
print(node.flow, "->", node.run_dir.relative_to(ws.run_root))
print("seed:", node.settings["seed"])
print("clocks:", dict(node.settings["clocks"]["main_clock"])["period"])
print("reads: ", {name: types for name, types in node.input_types.items()})
print("writes:", {name: types for name, types in node.output_types.items()})

`node.identity` is opaque text. It covers the effective settings, the parts of the design that the
node reads and where its inputs come from. Compare identities for equality only.

Each input says where its files come from: `producer` (another node writes the file), `source` (the
design supplies it) or `none` (an optional input has no file).

In [ ]:
for item in node.inputs:
    print(f"{item.name:12} {item.origin:9} {item.producer or '-':11} {item.reason}")

## Plan as data

`plan.to_dict()` is the document that `xeda run --dry-run --json` prints.

In [ ]:
document = plan.to_dict()
print(document["schema_version"], [entry["name"] for entry in document["nodes"]])
print(dict(plan.options))

## Run the plan

`ws.run(plan)` resolves the spec again under the locks and compares it with the plan. If nothing
differs, it runs the plan. Otherwise it raises `StalePlanError` (see below).

In [ ]:
result = ws.run(plan)
result.require_passed()
print(result.outcome, result.passed)
for name, node in result.nodes.items():
    print(f"{name:11} {node.outcome.value:7} {node.disposition.value}")

Plan again. The runs passed, so both nodes are fresh. A run does not change the identity of a node.

In [ ]:
after = ws.plan(spec)
print(*(f"{name}={node.state}" for name, node in after.nodes.items()))
print("same identities:", [n.identity for n in after.nodes.values()]
      == [n.identity for n in plan.nodes.values()])

A plan shows the effect of a change before you run. Another `seed` changes only `nextpnr`. Xeda
would reuse the run of Yosys.

In [ ]:
reseeded = Spec(
    "yosys_fpga+nextpnr",
    design,
    settings={
        "yosys_fpga": {"fpga.part": "LFE5U-85F-6BG381C", "clock.period": "10 ns"},
        "nextpnr": {"lpf_allow_unconstrained": True, "seed": 2},
    },
)
for name, node in ws.plan(reseeded).nodes.items():
    print(f"{name:11} {node.state:6} {node.reason}".rstrip())

## Force a rebuild

The options of a launch belong to the plan. Plan with the option, then run the plan. The plan shows
what the forced launch would do.

In [ ]:
forced = ws.plan(spec, rebuild_all=True)
print([(name, node.state, node.reason) for name, node in forced.nodes.items()])
print(ws.run(forced).require_passed().nodes["nextpnr"].disposition.value)

A plan carries its options, so `ws.run(plan)` takes none. Any launch option raises a `SpecError`
with the code `plan_options`, also an option that equals the one of the plan. With a spec, the
option goes to `run`.

In [ ]:
try:
    ws.run(plan, rebuild_all=True)
except SpecError as error:
    print(error.code)

## A plan belongs to its workspace

A plan belongs to the workspace that made it. Another workspace has another run root, so it has
other run directories. Running the plan there raises `StalePlanError`. The `difference` names the
first difference that Xeda found.

In [ ]:
other = Workspace(WORK / "xeda_run_other")
try:
    other.run(plan)
except StalePlanError as error:
    print(error.code)
    print(error.difference)

## Settings errors

A wrong request fails at planning. No tool has run. A problem with a setting raises a
`SettingsError`. It has the `code` of the first problem, and `errors` holds all of them. The helper
below prints one line for each problem.

In [ ]:
def check(flow, settings):
    try:
        ws.plan(Spec(flow, design, settings=settings))
    except SettingsError as error:
        for problem in error.errors:
            print(f"{problem.code} node={problem.node} setting={problem.setting}")
            print("   ", problem.problem.splitlines()[0])
            if problem.suggestions:
                print("    suggestions:", problem.suggestions)
    except SpecError as error:
        print(f"{error.code} ({type(error).__name__})")
        print("   ", error.message)

A setting needs the name of its flow. The error gives the right key.

In [ ]:
check("yosys_fpga+nextpnr", {"seed": 1})

A wrong setting name is refused. The error lists close names.

In [ ]:
check("yosys_fpga+nextpnr", {"nextpnr": {"sed": 1}})

A setting accepts exactly its type. The text `"1"` is not a whole number.

In [ ]:
check("yosys_fpga+nextpnr", {"nextpnr": {"seed": "1"}})

A physical quantity needs a unit. A bare number is refused, and so is a unit of another dimension.

In [ ]:
check("yosys_fpga+nextpnr", {"nextpnr.hold_detour_max": 10})
check("yosys_fpga+nextpnr", {"nextpnr.hold_detour_max": "10 MHz"})

A name that is not a node of the chain is refused. A nested table and a dotted key are the same
setting, so Xeda refuses a setting that you write twice, even with equal values.

In [ ]:
check("yosys_fpga", {"nextpnr": {"seed": 1}})
check("yosys_fpga+nextpnr", {"nextpnr": {"seed": 1}, "nextpnr.seed": 1})

One request can have several problems. The exception is the first one, and `errors` has all of
them.

In [ ]:
check("yosys_fpga+nextpnr", {"nextpnr": {"sed": 1, "seed": "1"}})

Xeda stores a quantity as canonical text. Two texts of one value give one value, and one identity.

In [ ]:
def hold(text):
    plan = ws.plan(Spec("yosys_fpga+nextpnr", design, settings={"nextpnr.hold_detour_max": text}))
    return plan.nodes["nextpnr"]

print(hold("6000 ps").settings["hold_detour_max"], hold("6000 ps").identity == hold("6 ns").identity)

A flow name with a typo raises `FlowNotFoundError`. A chain that cannot connect raises
`ChainError`. Both are subclasses of `SpecError`, so catch `SpecError` when you do not care which
one it is.

In [ ]:
for flow in ("yosys_fpgaa", "nextpnr+yosys_fpga"):
    try:
        ws.plan(Spec(flow, design))
    except SpecError as error:
        print(type(error).__name__, error.code)

## The module-level form

`xeda.plan` plans without a workspace object. It takes the run root as an argument.

In [ ]:
print(xeda.plan(spec, run_root=WORK / "xeda_run", local_settings=xeda.local_settings()).nodes["nextpnr"].state)

## Next

- A launch that does not pass, a time limit and a cancel: `02-simulation.ipynb`.
- Many points in one call: `03-sweep.ipynb`.